In [ ]:
# =============================================================================
# SECTION 0A — INSTALL
# =============================================================================
import subprocess
subprocess.run(["pip", "install", "torch-geometric", "-q"], check=True)
 
 
# =============================================================================
# SECTION 0B — CONFIG
# =============================================================================
import re, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data, Batch as PyGBatch
from torch_geometric.utils import dense_to_sparse
from sklearn.metrics import roc_auc_score
import pandas as pd
 
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
 
# --- Paths ---
ADJ_DIR     = Path("/kaggle/input/datasets/nhn2mm/chbmit-topk20")
FEAT_DIR    = Path("/kaggle/input/datasets/nhn2mm/chbmit-processed")
MODEL_PATH  = Path("/kaggle/input/datasets/nhn2mm/gae-joint-model/best_model_joint_lambda01.pt")
TEMP_DIR    = Path("/kaggle/input/datasets/nhn2mm/temporal-zscores")
GAMMA_DIR   = Path("/kaggle/input/datasets/nhn2mm/gamma-aec-scores")
SUMMARY_DIR = Path("/kaggle/input/datasets/nhn2mm/chb-mit-summary/summary")
OUT_DIR     = Path("/kaggle/working")
 
# --- Fixed split (seed=42, PERMANENT) ---
VAL_SUBJS  = ["chb10", "chb11", "chb22"]
TEST_SUBJS = ["chb03", "chb06", "chb13", "chb14",
              "chb15", "chb16", "chb17", "chb18"]
ALL_SUBJS  = VAL_SUBJS + TEST_SUBJS   # score both for TAU calibration
 
# --- Model architecture (LOCKED) ---
INPUT_DIM  = 23
HIDDEN_DIM = 64
LATENT_DIM = 16
N_CH       = 18
N_BANDS    = 5
LAMBDA     = 0.1
 
# --- Graph config ---
ADJS_SUFFIX = "_topk20"
 
# --- Ensemble weights (locked) ---
W_R, W_T, W_G = 0.35, 0.30, 0.35
 
# --- Threshold config ---
P95_PERCENTILE  = 95      # P95 of val subjects interictal z-scores
WIN_SEC         = 4       # seconds per window
MERGE_GAP_WIN   = 8       # merge gap for FDR/h counting (8 windows = 32s)
TOLERANCE_WIN   = 7       # ±7 windows = ±28s ~ ±30s clinical tolerance
TAU_Z_SWEEP     = [1.0, 1.5, 2.0, 2.5, 3.0]   # sweep for comparison table
 
BATCH_SIZE = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Test subjects:  {TEST_SUBJS}")
print(f"Val  subjects:  {VAL_SUBJS}  (TAU calibration only, no ictal labels used)")

In [ ]:
# =============================================================================
# SECTION 1A — MODEL DEFINITION (identical to CPD notebook)
# =============================================================================
class GAEEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(INPUT_DIM, HIDDEN_DIM)
        self.conv2 = GCNConv(HIDDEN_DIM, LATENT_DIM)
        self.relu  = nn.ReLU()
    def forward(self, x, ei, ea=None):
        return self.conv2(self.relu(self.conv1(x, ei, ea)), ei, ea)
 
class XDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM, 32), nn.ReLU(),
            nn.Linear(32, N_BANDS))
    def forward(self, z): return self.net(z)
 
class GAEModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder   = GAEEncoder()
        self.x_decoder = XDecoder()
 
print(f"GAEModel defined. Parameters: "
      f"{sum(p.numel() for p in GAEModel().parameters()):,}")
 
 
# =============================================================================
# SECTION 1B — LOAD MODEL + VERIFICATION
# =============================================================================
model = GAEModel().to(device)
model.load_state_dict(torch.load(str(MODEL_PATH), map_location=device))
model.eval()
 
_b = model.encoder.conv1.bias
if _b is None or _b.abs().max().item() < 0.005:
    raise RuntimeError("Random init detected — wrong model path.")
print(f"Model loaded. Bias max = {_b.abs().max().item():.4f}  (expect ~0.8676)")

In [ ]:
# =============================================================================
# SECTION 2 — GAE RECONSTRUCTION SCORING (val + test subjects)
# Score ALL_SUBJS = VAL + TEST so we can calibrate TAU on val interictal.
# =============================================================================
def score_adj_files(adj_path, feat_path):
    adjs  = np.load(adj_path,  mmap_mode='r')
    feats = np.load(feat_path, mmap_mode='r')
    scores = []
    model.eval()
    with torch.no_grad():
        for s in range(0, len(adjs), BATCH_SIZE):
            e   = min(s + BATCH_SIZE, len(adjs))
            B   = e - s
            A   = torch.tensor(adjs[s:e].astype(np.float32),  device=device)
            Xt  = torch.tensor(feats[s:e].astype(np.float32), device=device)
            An  = A / (A.amax(dim=(1,2), keepdim=True) + 1e-8)
            Xmn = Xt.amin(dim=1, keepdim=True)
            Xmx = Xt.amax(dim=1, keepdim=True)
            Xn  = (Xt - Xmn) / (Xmx - Xmn + 1e-8)
            dl  = [Data(x=torch.cat([An[b], Xn[b]], dim=1),
                        edge_index=dense_to_sparse(A[b])[0],
                        edge_attr =dense_to_sparse(A[b])[1]) for b in range(B)]
            pg  = PyGBatch.from_data_list(dl).to(device)
            z   = model.encoder(pg.x, pg.edge_index, pg.edge_attr)
            zpg = z.view(B, N_CH, LATENT_DIM)
            Ah  = torch.clamp(torch.bmm(zpg, zpg.transpose(1,2)), 0., 1.)
            Xh  = model.x_decoder(z).view(B, N_CH, N_BANDS)
            sc  = ((A - Ah)**2).mean(dim=(1,2)) + \
                  LAMBDA * ((Xn - Xh)**2).mean(dim=(1,2))
            scores.extend(sc.cpu().numpy().tolist())
    return np.array(scores, dtype=np.float32)
 
 
print("\n" + "=" * 60)
print("SECTION 2: GAE Reconstruction Scoring (val + test subjects)")
print("=" * 60)
 
raw_recon_inter = {}
raw_recon_ictal = {}
 
for subj in ALL_SUBJS:
    ai = str(ADJ_DIR  / f"{subj}_interictal_adjs{ADJS_SUFFIX}.npy")
    fi = str(FEAT_DIR / f"{subj}_interictal_features.npy")
    ac = str(ADJ_DIR  / f"{subj}_ictal_adjs{ADJS_SUFFIX}.npy")
    fc = str(FEAT_DIR / f"{subj}_ictal_features.npy")
 
    raw_recon_inter[subj] = score_adj_files(ai, fi)
    # Val subjects không cần ictal scores (TAU calibration không dùng ictal)
    if subj in TEST_SUBJS:
        raw_recon_ictal[subj] = score_adj_files(ac, fc)
 
    role = "VAL" if subj in VAL_SUBJS else "TEST"
    print(f"  [{role}] {subj}: inter={len(raw_recon_inter[subj])}"
          + (f", ictal={len(raw_recon_ictal[subj])}"
             if subj in TEST_SUBJS else ""))
 
print("Section 2 complete.")

In [ ]:
# =============================================================================
# SECTION 3 — TEMPORAL LSTM SCORES (test subjects only)
# =============================================================================
print("\n" + "=" * 60)
print("SECTION 3: Temporal LSTM Scores (test subjects only)")
print("=" * 60)

raw_temp_inter = {}
raw_temp_ictal = {}

for subj in TEST_SUBJS:          # chỉ test subjects — val không có file này
    raw_temp_inter[subj] = np.load(
        str(TEMP_DIR / f"temporal_{subj}_zinter.npy")).astype(np.float32)
    raw_temp_ictal[subj] = np.load(
        str(TEMP_DIR / f"temporal_{subj}_zictal.npy")).astype(np.float32)
    print(f"  [TEST] {subj}: inter={len(raw_temp_inter[subj])}, "
          f"ictal={len(raw_temp_ictal[subj])}")

print("Section 3 complete.")


# =============================================================================
# SECTION 4 — GAMMA AEC SCORES (test subjects only)
# =============================================================================
print("\n" + "=" * 60)
print("SECTION 4: Gamma AEC Scores (test subjects only)")
print("=" * 60)

raw_gamma_inter = {}
raw_gamma_ictal = {}

for subj in TEST_SUBJS:          # chỉ test subjects — val không có file này
    raw_gamma_inter[subj] = np.load(
        str(GAMMA_DIR / f"gamma_aec_{subj}_inter.npy")).astype(np.float32)
    raw_gamma_ictal[subj] = np.load(
        str(GAMMA_DIR / f"gamma_aec_{subj}_ictal.npy")).astype(np.float32)
    print(f"  [TEST] {subj}: inter={len(raw_gamma_inter[subj])}, "
          f"ictal={len(raw_gamma_ictal[subj])}")

print("Section 4 complete.")


# =============================================================================
# SECTION 5A — ALL-WINDOW Z-NORMALIZATION
#
# Val subjects: chỉ z-norm reconstruction scores (temporal/gamma không có)
# Test subjects: z-norm cả 3 signals độc lập
#
# TAU calibration (Section 5C) dùng z_recon_inter của val subjects.
# Sau z-norm, reconstruction z-scores của val subjects có distribution
# tương đương ensemble z-scores (mean≈0, std≈1).
# =============================================================================
def robust_z_norm(raw_inter, raw_ictal=None):
    """
    Z-norm dùng median/MAD từ ALL windows pooled.
    Nếu raw_ictal=None: chỉ dùng interictal (cho val subjects).
    """
    if raw_ictal is not None:
        all_s = np.concatenate([raw_inter, raw_ictal])
    else:
        all_s = raw_inter
    med = np.median(all_s)
    mad = np.median(np.abs(all_s - med)) + 1e-9
    z_inter = (raw_inter - med) / mad
    z_ictal = (raw_ictal - med) / mad if raw_ictal is not None else None
    return z_inter, z_ictal


print("\n" + "=" * 60)
print("SECTION 5A: All-window Z-normalization")
print("=" * 60)

z_recon_inter = {}; z_recon_ictal = {}
z_temp_inter  = {}; z_temp_ictal  = {}
z_gamma_inter = {}; z_gamma_ictal = {}

# Val subjects: chỉ z-norm reconstruction (không có temporal/gamma)
for subj in VAL_SUBJS:
    z_recon_inter[subj], _ = robust_z_norm(raw_recon_inter[subj], None)
    print(f"  [VAL]  {subj}: recon z-norm applied (reconstruction only)")

# Test subjects: z-norm cả 3 signals
for subj in TEST_SUBJS:
    z_recon_inter[subj], z_recon_ictal[subj] = robust_z_norm(
        raw_recon_inter[subj], raw_recon_ictal[subj])

    z_temp_inter[subj], z_temp_ictal[subj] = robust_z_norm(
        raw_temp_inter[subj], raw_temp_ictal[subj])

    z_gamma_inter[subj], z_gamma_ictal[subj] = robust_z_norm(
        raw_gamma_inter[subj], raw_gamma_ictal[subj])

    print(f"  [TEST] {subj}: all 3 signals z-norm applied")

print("Section 5A complete.")


# =============================================================================
# SECTION 5B — 3-WAY ENSEMBLE (test subjects only)
# Val subjects dùng reconstruction z-scores đơn thuần cho TAU calibration
# =============================================================================
print("\n" + "=" * 60)
print(f"SECTION 5B: 3-way ensemble  W_R={W_R}, W_T={W_T}, W_G={W_G}")
print("=" * 60)

ens_inter = {}
ens_ictal = {}

# Val subjects: dùng reconstruction z-score làm proxy cho ensemble
# (sau z-norm, scale tương đương; P95 của reconstruction ≈ P95 của ensemble)
for subj in VAL_SUBJS:
    ens_inter[subj] = z_recon_inter[subj].copy()
    print(f"  [VAL]  {subj}: using recon z-scores for TAU calibration "
          f"({len(ens_inter[subj])} windows)")

# Test subjects: full 3-way ensemble
for subj in TEST_SUBJS:
    ni = min(len(z_recon_inter[subj]),
             len(z_temp_inter[subj]),
             len(z_gamma_inter[subj]))
    nc = min(len(z_recon_ictal[subj]),
             len(z_temp_ictal[subj]),
             len(z_gamma_ictal[subj]))

    ens_inter[subj] = (W_R * z_recon_inter[subj][:ni] +
                       W_T * z_temp_inter[subj][:ni]  +
                       W_G * z_gamma_inter[subj][:ni])
    ens_ictal[subj] = (W_R * z_recon_ictal[subj][:nc] +
                       W_T * z_temp_ictal[subj][:nc]  +
                       W_G * z_gamma_ictal[subj][:nc])

    mzi = float(np.median(ens_ictal[subj]))
    print(f"  [TEST] {subj}: inter={len(ens_inter[subj])}, "
          f"ictal={len(ens_ictal[subj])}, mzi={mzi:.3f}")

print("Section 5B complete.")
print()
print("NOTE: TAU will be calibrated on VAL subjects' reconstruction z-scores.")
print("      This is conservative (reconstruction-only scale) but fully")
print("      unsupervised. If TAU seems too low/high after Section 5C,")
print("      consider using val reconstruction scores × correction factor.")

In [ ]:
# =============================================================================
# SECTION 5C — THRESHOLD CALIBRATION (P95 of val interictal — fully unsupervised)
#
# TAU = percentile(ens_inter_val_all_pooled, 95)
# Hoàn toàn unsupervised: chỉ dùng interictal windows của val subjects,
# không dùng bất kỳ ictal label nào.
#
# Ý nghĩa: model flags window as anomalous nếu reconstruction error vượt
# 95th percentile của normal EEG reconstruction error.
# =============================================================================
print("\n" + "=" * 60)
print("SECTION 5C: Threshold Calibration (P95 val interictal)")
print("=" * 60)
 
val_inter_pooled = np.concatenate([ens_inter[s] for s in VAL_SUBJS])
TAU = float(np.percentile(val_inter_pooled, P95_PERCENTILE))
 
print(f"Val interictal pooled: {len(val_inter_pooled):,} windows")
print(f"  P50 = {np.percentile(val_inter_pooled, 50):.4f}")
print(f"  P90 = {np.percentile(val_inter_pooled, 90):.4f}")
print(f"  P95 = {np.percentile(val_inter_pooled, 95):.4f}  ← TAU")
print(f"  P99 = {np.percentile(val_inter_pooled, 99):.4f}")
print(f"\nTAU (P95 val interictal ensemble) = {TAU:.4f}")
 
# Guard: TAU phải hợp lý (không quá cao không quá thấp)
if TAU < 0.0 or TAU > 5.0:
    print(f"WARNING: TAU={TAU:.4f} out of expected range [0.0, 5.0].")
    print("         Check if z-norm was applied correctly.")
else:
    print("TAU check passed.")
 
# Convert TAU to tau_z scale (vì đây là z-normalized scores, TAU đã là tau_z)
# TAU ≈ 2.0 là expected sau z-norm (P95 ≈ 1.96 cho Gaussian)
print(f"\nNote: After z-norm, P95 ≈ 2.0 for Gaussian baseline.")
print(f"      Actual P95 = {TAU:.4f} — may differ due to non-Gaussian EEG distribution.")

In [ ]:
# =============================================================================
# SECTION 6A — AUROC VERIFICATION (threshold-independent)
# =============================================================================
print("\n" + "=" * 60)
print("SECTION 6A: AUROC Verification (threshold-independent)")
print("=" * 60)
 
auroc_rows = []
per_subject_aurocs = []
 
print(f"{'Subj':<6} {'AUROC_ens':>10} {'AUROC_recon':>12} "
      f"{'AUROC_temp':>11} {'AUROC_gamma':>12} {'mzi':>7}")
print("-" * 62)
 
for subj in TEST_SUBJS:
    ni = len(ens_inter[subj])
    nc = len(ens_ictal[subj])
    y  = np.concatenate([np.zeros(ni), np.ones(nc)])
 
    auc_ens   = roc_auc_score(y,
        np.concatenate([ens_inter[subj], ens_ictal[subj]]))
    auc_recon = roc_auc_score(
        np.concatenate([np.zeros(len(z_recon_inter[subj])),
                        np.ones( len(z_recon_ictal[subj]))]),
        np.concatenate([z_recon_inter[subj], z_recon_ictal[subj]]))
    auc_temp  = roc_auc_score(y,
        np.concatenate([z_temp_inter[subj][:ni],
                        z_temp_ictal[subj][:nc]]))
    auc_gamma = roc_auc_score(y,
        np.concatenate([z_gamma_inter[subj][:ni],
                        z_gamma_ictal[subj][:nc]]))
    mzi = float(np.median(ens_ictal[subj]))
 
    print(f"{subj:<6} {auc_ens:>10.4f} {auc_recon:>12.4f} "
          f"{auc_temp:>11.4f} {auc_gamma:>12.4f} {mzi:>7.3f}")
 
    per_subject_aurocs.append(auc_ens)
    auroc_rows.append({
        "subject":     subj,
        "auroc_ens":   round(auc_ens,   4),
        "auroc_recon": round(auc_recon, 4),
        "auroc_temp":  round(auc_temp,  4),
        "auroc_gamma": round(auc_gamma, 4),
        "mzi_ens":     round(mzi,       4),
        "n_inter":     ni,
        "n_ictal":     nc,
    })
 
macro_auroc = float(np.mean(per_subject_aurocs))
print(f"\nMACRO AUROC: {macro_auroc:.4f}  (expected ~0.7957)")
 
df_auroc = pd.DataFrame(auroc_rows)
df_auroc.to_csv(OUT_DIR / "threshold_auroc.csv", index=False)
print("Saved: threshold_auroc.csv")
 
 
# =============================================================================
# SECTION 6B — WINDOW-LEVEL METRICS AT TAU
# Sensitivity = TP_window / (TP_window + FN_window)
# Specificity = TN_window / (TN_window + FP_window)
# =============================================================================
print("\n" + "=" * 60)
print(f"SECTION 6B: Window-level metrics at TAU={TAU:.4f}")
print("=" * 60)
 
print(f"{'Subj':<6} {'Sensitivity':>12} {'Specificity':>12} "
      f"{'Precision':>10} {'TP_w':>6} {'FN_w':>6} {'FP_w':>6} {'TN_w':>6}")
print("-" * 72)
 
window_rows = []
for subj in TEST_SUBJS:
    z_i = ens_inter[subj]
    z_c = ens_ictal[subj]
 
    tp_w = int((z_c >  TAU).sum())
    fn_w = int((z_c <= TAU).sum())
    fp_w = int((z_i >  TAU).sum())
    tn_w = int((z_i <= TAU).sum())
 
    sens = tp_w / max(tp_w + fn_w, 1)
    spec = tn_w / max(tn_w + fp_w, 1)
    prec = tp_w / max(tp_w + fp_w, 1)
 
    print(f"{subj:<6} {sens:>12.4f} {spec:>12.4f} "
          f"{prec:>10.4f} {tp_w:>6} {fn_w:>6} {fp_w:>6} {tn_w:>6}")
 
    window_rows.append({
        "subject": subj, "tau": round(TAU, 4),
        "sensitivity": round(sens, 4), "specificity": round(spec, 4),
        "precision":   round(prec, 4),
        "tp_w": tp_w, "fn_w": fn_w, "fp_w": fp_w, "tn_w": tn_w,
    })
 
macro_sens = np.mean([r["sensitivity"] for r in window_rows])
macro_spec = np.mean([r["specificity"] for r in window_rows])
print(f"{'MACRO':<6} {macro_sens:>12.4f} {macro_spec:>12.4f}")
 
df_window = pd.DataFrame(window_rows)
df_window.to_csv(OUT_DIR / "threshold_window_metrics.csv", index=False)
print("Saved: threshold_window_metrics.csv")
 
 
# =============================================================================
# SECTION 6C — EVENT-LEVEL EVALUATION AT TAU (PRIMARY CLINICAL METRIC)
#
# TP_event: seizure has >= 1 window flagged above TAU
# FN_event: seizure has 0 windows flagged
# FDR/h: number of distinct false-positive clusters per interictal hour
# Latency: time from seizure onset to first flagged window (seconds)
# Merge gap: 8 windows = 32 seconds (same as CPD notebook)
# =============================================================================
def parse_seizures(subj):
    """
    Parse seizure onset/offset from CHB-MIT summary files.
    Returns list of (onset_s, offset_s) tuples.
    """
    path = SUMMARY_DIR / f"{subj}-summary.txt"
    text = Path(path).read_text()
    starts = [int(x) for x in re.findall(
        r'Seizure.*?Start Time.*?:\s*(\d+)\s*second', text, re.I)]
    ends   = [int(x) for x in re.findall(
        r'Seizure.*?End Time.*?:\s*(\d+)\s*second',   text, re.I)]
    seizures = list(zip(starts, ends))
    return seizures
 
 
def event_eval(z_inter, z_ictal, seizure_list, tau,
               win_sec=WIN_SEC, merge_gap_win=MERGE_GAP_WIN):
    """
    Event-level evaluation.
 
    TP logic: for each seizure, take the corresponding ictal windows in order.
              If ANY window exceeds tau → TP. Latency = first flagged window.
    FDR/h:   count distinct clusters of flagged interictal windows
              (merge_gap_win = 8 windows = 32s merge gap).
 
    Returns: (tp, fn, fdr_h, mean_latency_s)
    """
    n_ictal_total = len(z_ictal)
    ptr, tp, fn   = 0, 0, 0
    latencies     = []
 
    for (onset_s, offset_s) in seizure_list:
        n_win = max(1, int(np.ceil((offset_s - onset_s) / win_sec)))
        # Guard against running out of ictal windows
        if ptr + n_win > n_ictal_total:
            n_win = max(0, n_ictal_total - ptr)
        if n_win == 0:
            fn += 1
            continue
 
        wins = z_ictal[ptr: ptr + n_win]
        if (wins > tau).any():
            tp += 1
            # Latency: first flagged window index × win_sec
            first_idx = int(np.argmax(wins > tau))
            latencies.append((first_idx + 1) * win_sec)
        else:
            fn += 1
        ptr += n_win
 
    # FDR/h from interictal windows
    n_hours  = len(z_inter) * win_sec / 3600.0
    flagged  = z_inter > tau
    n_events, in_ev, gap = 0, False, 0
    for d in flagged:
        if d:
            if not in_ev:
                n_events += 1
                in_ev = True
            gap = 0
        elif in_ev:
            gap += 1
            if gap > merge_gap_win:
                in_ev = False
                gap = 0
    fdr_h    = n_events / max(n_hours, 1e-6)
    mean_lat = float(np.mean(latencies)) if latencies else float('nan')
    return tp, fn, fdr_h, mean_lat
 
 
print("\n" + "=" * 60)
print(f"SECTION 6C: Event-level evaluation at TAU={TAU:.4f} (P95 val interictal)")
print("Fully unsupervised — no ictal labels used in threshold calibration")
print("=" * 60)
 
print(f"\n{'Subj':<6} {'GT':>4} {'TP':>4} {'FN':>4} "
      f"{'EvSens':>8} {'FDR/h':>7} {'Lat_s':>7}")
print("-" * 48)
 
event_rows = []
total_gt, total_tp, total_fn = 0, 0, 0
all_fdr, all_lat = [], []
 
for subj in TEST_SUBJS:
    seizures = parse_seizures(subj)
    gt       = len(seizures)
    tp, fn, fdr_h, mean_lat = event_eval(
        ens_inter[subj], ens_ictal[subj], seizures, TAU)
 
    ev_sens    = tp / max(gt, 1)
    total_gt  += gt
    total_tp  += tp
    total_fn  += fn
    all_fdr.append(fdr_h)
    if not np.isnan(mean_lat):
        all_lat.append(mean_lat)
 
    lat_str = f"{mean_lat:.1f}" if not np.isnan(mean_lat) else "—"
    print(f"{subj:<6} {gt:>4} {tp:>4} {fn:>4} "
          f"{ev_sens:>8.3f} {fdr_h:>7.1f} {lat_str:>7}")
 
    event_rows.append({
        "subject":    subj,
        "tau":        round(TAU, 4),
        "gt":         gt,
        "tp":         tp,
        "fn":         fn,
        "ev_sens":    round(ev_sens,  4),
        "fdr_h":      round(fdr_h,   2),
        "mean_lat_s": round(mean_lat, 1) if not np.isnan(mean_lat) else None,
    })
 
macro_ev  = total_tp / max(total_gt, 1)
mean_fdr  = float(np.nanmean(all_fdr))
mean_lat  = float(np.nanmean(all_lat)) if all_lat else float('nan')
lat_str   = f"{mean_lat:.1f}" if not np.isnan(mean_lat) else "—"
 
print("-" * 48)
print(f"{'MACRO':<6} {total_gt:>4} {total_tp:>4} {total_fn:>4} "
      f"{macro_ev:>8.3f} {mean_fdr:>7.1f} {lat_str:>7}")
print(f"\nSummary: {total_tp}/{total_gt} seizures detected = {macro_ev:.1%}")
print(f"         FDR/h = {mean_fdr:.1f}")
print(f"         Mean onset latency = {lat_str}s")
 
df_event = pd.DataFrame(event_rows)
df_event.to_csv(OUT_DIR / "threshold_event_metrics.csv", index=False)
print("Saved: threshold_event_metrics.csv")
 
 
# =============================================================================
# SECTION 6D — TAU SWEEP (for trade-off table in Chapter 3)
# Test multiple tau_z values to show sensitivity vs FDR/h trade-off.
# Useful for Chapter 4 Discussion: why P95 was chosen.
# =============================================================================
print("\n" + "=" * 60)
print("SECTION 6D: Tau sweep (sensitivity vs FDR/h trade-off)")
print("=" * 60)
 
# Compute sigma for each subject (for tau_z → tau conversion)
# tau = median_inter + tau_z * MAD_inter
# But since we're already z-normalized, tau_z directly applies to z-scores
# val TAU = P95 ≈ some tau_z value; we also sweep fixed tau_z values
 
# First, compute what tau_z our TAU corresponds to
# (since z-norm was applied, TAU is already in z-score units)
TAU_Z_P95 = TAU  # after z-norm, TAU is directly a tau_z value
print(f"P95 threshold in z-score units: tau_z = {TAU_Z_P95:.4f}\n")
 
print(f"{'tau_z':>6} {'TP':>4} {'GT':>4} {'EvSens':>8} "
      f"{'FDR/h':>7} {'Lat_s':>7}")
print("-" * 45)
 
sweep_rows = []
for tau_z in TAU_Z_SWEEP:
    t_gt, t_tp, t_fn = 0, 0, 0
    t_fdr, t_lat     = [], []
 
    for subj in TEST_SUBJS:
        seizures = parse_seizures(subj)
        tp, fn, fdr_h, mean_lat = event_eval(
            ens_inter[subj], ens_ictal[subj], seizures, tau_z)
        t_gt  += len(seizures)
        t_tp  += tp
        t_fn  += fn
        t_fdr.append(fdr_h)
        if not np.isnan(mean_lat):
            t_lat.append(mean_lat)
 
    ev  = t_tp / max(t_gt, 1)
    fdr = float(np.nanmean(t_fdr))
    lat = float(np.nanmean(t_lat)) if t_lat else float('nan')
    lat_s = f"{lat:.1f}" if not np.isnan(lat) else "—"
 
    marker = " ← P95" if abs(tau_z - TAU_Z_P95) < 0.05 else ""
    print(f"{tau_z:>6.1f} {t_tp:>4} {t_gt:>4} {ev:>8.3f} "
          f"{fdr:>7.1f} {lat_s:>7}{marker}")
 
    sweep_rows.append({
        "tau_z": tau_z, "tp": t_tp, "gt": t_gt,
        "ev_sens": round(ev, 4), "fdr_h": round(fdr, 2),
        "mean_lat_s": round(lat, 1) if not np.isnan(lat) else None,
    })
 
df_sweep = pd.DataFrame(sweep_rows)
df_sweep.to_csv(OUT_DIR / "threshold_tau_sweep.csv", index=False)
print("\nSaved: threshold_tau_sweep.csv")

In [ ]:
# =============================================================================
# SECTION 7 — FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 60)
print("FINAL SUMMARY — THRESHOLD PIPELINE (P95 Unsupervised)")
print("=" * 60)
print(f"\nMacro AUROC (ensemble, threshold-independent): {macro_auroc:.4f}")
print(f"TAU (P95 val interictal):                       {TAU:.4f}")
print(f"\nEvent-level at P95:")
print(f"  Event sensitivity: {total_tp}/{total_gt} = {macro_ev:.1%}")
print(f"  Mean FDR/h:        {mean_fdr:.1f}")
print(f"  Mean latency:      {lat_str}s")
print(f"\nWindow-level at P95:")
print(f"  Macro sensitivity: {macro_sens:.4f}")
print(f"  Macro specificity: {macro_spec:.4f}")
print(f"\nOutput files:")
print(f"  threshold_auroc.csv          → AUROC per subject")
print(f"  threshold_window_metrics.csv → window sens/spec at P95")
print(f"  threshold_event_metrics.csv  → event TP/FN/FDR/h/latency at P95")
print(f"  threshold_tau_sweep.csv      → trade-off curve")
print(f"\nComparison with CPD (from cpd_results_v12_combined.csv):")
print(f"  CPD pen=0.5: DR=75.0%, FDR/h=24.4, latency=-2.2s, no ictal labels needed")
print(f"  Threshold P95: see results above")
print(f"\nKey thesis argument:")
print(f"  Threshold-based detection requires fixed TAU calibrated on normal EEG.")
print(f"  CPD is direction-agnostic (detects both up/down shifts → helps chb06)")
print(f"  and adapts to signal distribution via BIC penalty instead of fixed TAU.")